In [ ]:
# ============================================================
# OpenQSim Phase 0A — Full sweep (GPU + CPU working together)
# ============================================================
# HOW TO USE:
#   1. Start a new Kaggle session with GPU T4 accelerator
#      (Settings → Accelerator → GPU T4 x2)
#   2. Provide keys via Kaggle Secrets OR paste into the _KEYS block below:
#      KAGGLE_USERNAME, KAGGLE_KEY  (required for push)
#      NVIDIA_API_KEY  (optional: NVIDIA_API_KEY_1, GROQ_API_KEY)
#   3. Dataset: harshalekkala1/openqsim-phase0a (already created)
#   4. Run cells top to bottom — sweeps all 1320 combos from index 0
#
# HOW GPU + CPU SHARE THE WORK:
#   aer_statevector → GPU when Aer's GPU kernels actually run on this card.
#   aer_mps         → CPU  (matrix_product_state is CPU-only in Aer 0.15.1)
#   Cell 5 runs a real GPU kernel; if it fails (e.g. Kaggle T4 →
#   cudaErrorNoKernelImageForDevice) the whole sweep falls back to CPU via
#   OPENQSIM_DEVICE=CPU. Detection alone is NOT trusted.
#
# VERSION NOTE: Pinned to qiskit==1.4.2 (last 1.x). Do NOT loosen this:
#   Aer 0.15.1 needs convert_to_target (removed in Qiskit 2.0) and
#   qft.py uses the QFT class (removed in Qiskit 2.1).
# ============================================================

import sys, subprocess, os, shutil, importlib.util, re, json
from pathlib import Path

# ── 0. Keys (optional) ─────────────────────────────────────
# Leave blank to use Kaggle Secrets (cell 3) instead. Paste here only
# if you are NOT using Secrets. Anything non-blank is used as-is.
#   KAGGLE_*    → dataset push (required somewhere: here OR Secrets)
#   NVIDIA/GROQ → optional LLM advisor predictions; all blank = advisor off
_KEYS = {
    "KAGGLE_USERNAME":  "",
    "KAGGLE_KEY":       "",
    "NVIDIA_API_KEY":   "",
    "NVIDIA_API_KEY_1": "",   # second key for parallel throughput
    "GROQ_API_KEY":     "",   # fallback if NVIDIA unset
}
for _k, _v in _KEYS.items():
    if _v.strip():
        os.environ[_k] = _v.strip()
if os.environ.get("NVIDIA_API_KEY_1"):
    os.environ["NVIDIA_API_KEY_COUNT"] = "2"

# ── 1. Install dependencies ────────────────────────────────
print("[1/5] Installing dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y", "-q",
    "qiskit", "qiskit-aer", "qiskit-aer-gpu", "qiskit-terra",
], check=False)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "qiskit==1.4.2",          # last 1.x: exports convert_to_target, has QFT class
    "qiskit-aer-gpu==0.15.1", # GPU build — also works for CPU backends
    "pynvml", "nvidia-ml-py", "pyyaml",
    "pandas", "kaggle>=2.2.2", "requests",
    "python-dotenv", "networkx",
])
print("      Done.")

# Report Qiskit/Aer + device detection. NOTE: 'GPU' in available_devices()
# only means Aer was *built* with GPU support — the kernels may still be
# unrunnable on this card. Cell 5 runs a real kernel to decide; this is info.
import qiskit, qiskit_aer
from qiskit_aer import AerSimulator
print(f"      Qiskit: {qiskit.__version__}  Aer: {qiskit_aer.__version__}")
devices = AerSimulator().available_devices()
print(f"      Aer devices: {devices}")
print("      GPU backend detected" if "GPU" in devices else "      GPU backend not detected")

In [ ]:
# ── 2. Clone latest code ──────────────────────────────────
print("[2/5] Cloning repository...")
REPO_URL = "https://github.com/lekkalaharsha/opensim-ai"
REPO_DIR = "/kaggle/working/opensim-ai"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
print("      Done.")
sys.path.insert(0, REPO_DIR)

In [ ]:
# ── 3. Load secrets + pick sweep ──────────────────────────
print("[3/5] Loading secrets...")
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "NVIDIA_API_KEY",
                "NVIDIA_API_KEY_1", "GROQ_API_KEY"):
        try:
            _v = _s.get_secret(_k)
            if _v:
                os.environ[_k] = _v
                print(f"      [OK] {_k} loaded")
        except Exception:
            pass
except Exception as e:
    print(f"      WARN secrets: {e}")

if os.environ.get("NVIDIA_API_KEY_1"):
    os.environ["NVIDIA_API_KEY_COUNT"] = "2"
    print("      [OK] Dual NVIDIA key mode")

# ── Which sweep this session? ─────────────────────────────
#   sweep_config_small.yaml  → 8–16q   (this session)
#   sweep_config_medium.yaml → 16–24q  (next session)
#   sweep_config_0a.yaml     → full 1320 (only once GPU works)
SWEEP_NAME = "sweep_config_small.yaml"

# CPU-only: Kaggle T4 GPU kernels are unusable (cudaErrorNoKernelImageForDevice).
# Set OPENQSIM_DEVICE before any backend is built so AerConfig pins CPU.
os.environ["OPENQSIM_DEVICE"] = "CPU"

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "harshalekkala1")
KAGGLE_DATASET  = f"{KAGGLE_USERNAME}/openqsim-phase0a"
OUTPUT_DIR      = "/kaggle/working/benchmark_outputs"
SWEEP_CONFIG    = f"{REPO_DIR}/benchmark/{SWEEP_NAME}"
print(f"      Sweep: {SWEEP_NAME}  |  device: CPU")
print(f"      Target dataset: {KAGGLE_DATASET}")

In [ ]:
# ── 4. Patch imports + load modules ──────────────────────
print("[4/5] Patching imports...")
KAGGLE_MOD_DIR = f"{REPO_DIR}/kaggle"

def _patch(path):
    txt = open(path).read()
    txt = re.sub(r'from kaggle\.(\w+) import', r'from \1 import', txt)
    txt = re.sub(r'import kaggle\.(\w+)', r'import \1', txt)
    open(path, "w").write(txt)

for py in Path(KAGGLE_MOD_DIR).glob("*.py"):
    _patch(str(py))
_runner_py = f"{REPO_DIR}/benchmark/runner.py"
if os.path.exists(_runner_py):
    _patch(_runner_py)

sys.path.insert(0, KAGGLE_MOD_DIR)

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_r = _load("kaggle_runner",    f"{KAGGLE_MOD_DIR}/runner.py")
_a = _load("kaggle_assembler", f"{KAGGLE_MOD_DIR}/dataset_assembler.py")
_c = _load("kaggle_client",    f"{KAGGLE_MOD_DIR}/api_client.py")
KaggleRunner     = _r.KaggleRunner
DatasetAssembler = _a.DatasetAssembler
KaggleAPIClient  = _c.KaggleAPIClient

from backend.environment import collect_environment_metadata
env = collect_environment_metadata()
print(f"      GPU: {env.gpu_name} ({env.gpu_memory_mb} MB)")
print(f"      Qiskit: {env.qiskit_version}")

devices = AerSimulator().available_devices()
print(f"      Aer devices: {devices}")
print("      [OK] aer_statevector → GPU | aer_mps → CPU")

if KaggleAPIClient(dataset=KAGGLE_DATASET).dataset_exists():
    print(f"      [OK] Dataset exists: {KAGGLE_DATASET}")
else:
    print(f"      WARN Dataset not found: {KAGGLE_DATASET}")
    print(f"           Create it at kaggle.com/datasets/add — will zip as fallback")
print("      Modules loaded.")

In [ ]:
# ── 5. Run sweep ─────────────────────────────────────────
print("[5/5] Running benchmark sweep...")

if os.environ.get("OPENQSIM_DEVICE") == "CPU":
    print("      Device pinned to CPU (cell 3) — skipping GPU canary")
else:
    # GPU canary — actually run a kernel. 'GPU' in available_devices() lies on
    # Kaggle T4 (cudaErrorNoKernelImageForDevice). On failure pin the whole sweep
    # to CPU so it doesn't die one aer_statevector combo at a time.
    GPU_OK = False
    try:
        from qiskit import QuantumCircuit
        _qc = QuantumCircuit(20)
        _qc.h(range(20))
        _qc.save_statevector()
        _res = AerSimulator(method="statevector", device="GPU").run(_qc).result()
        GPU_OK = _res.success
    except Exception as e:
        print(f"      GPU canary error: {e}")
    if GPU_OK:
        print("      [OK] GPU kernels run — aer_statevector → GPU")
    else:
        os.environ["OPENQSIM_DEVICE"] = "CPU"
        print("      WARN GPU kernels unrunnable — pinning sweep to CPU")

runner = KaggleRunner(
    sweep_config_path=SWEEP_CONFIG,
    output_dir=OUTPUT_DIR,
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset=KAGGLE_DATASET,
    use_advisor=True,
)
sweep_result = runner.run()
print(f"\n      Sweep done: {sweep_result.completed_count}/{sweep_result.total_combinations}")
print(f"      OOM: {sweep_result.oom_count}  Errors: {sweep_result.error_count}")
print(f"      Time: {sweep_result.duration_seconds/3600:.1f}h")

In [ ]:
# ── 6. Assemble dataset ───────────────────────────────────
print("\nAssembling dataset...")
assembler = DatasetAssembler(
    raw_dir=f"{OUTPUT_DIR}/raw",
    dataset_dir=f"{OUTPUT_DIR}/datasets/openqsim_v0.1",
)
manifest = assembler.assemble()
print(f"      {manifest.records} records | {manifest.successful_runs} successful")
print(f"      Backends: {manifest.backends} | Qubits: {manifest.qubit_range}")

In [ ]:
# ── 7. Push to Kaggle (zip fallback on 403/404) ──────────
print("\nPushing dataset...")
dataset_dir = Path(OUTPUT_DIR)
(dataset_dir / "dataset-metadata.json").write_text(json.dumps({
    "title": "OpenQSim Phase 0A Benchmark Outputs",
    "id": KAGGLE_DATASET,
    "licenses": [{"name": "MIT"}],
}))
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(dataset_dir),
     "-m", f"Phase 0A complete: {manifest.records} records"],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print(f"[OK] Pushed to {KAGGLE_DATASET}")
else:
    zip_path = shutil.make_archive(
        "/kaggle/working/benchmark_outputs_backup", "zip", str(OUTPUT_DIR)
    )
    print(f"WARN Push failed: {r.stderr[:200]}")
    print(f"[OK] Zip saved: {zip_path} — download from Output tab")

print("\n" + "=" * 50)
print("OPENQSIM PHASE 0A SWEEP COMPLETE")
print("=" * 50)
print(f"  Records:   {manifest.records}")
print(f"  Successes: {manifest.successful_runs}")
print(f"  Backends:  {manifest.backends}")
print(f"  Qubits:    {manifest.qubit_range}")
print(f"  Depths:    {manifest.depth_range}")
print(f"  Dataset:   {KAGGLE_DATASET}")
print("=" * 50)